In [ ]:
# ============================================================
# 🧩 STEP 1: Environment Setup
# ============================================================
!pip install -q transformers accelerate datasets evaluate rouge-score bert-score bitsandbytes torch torchvision torchaudio

from google.colab import userdata
from huggingface_hub import login

# 🔐 Login to Hugging Face (set token in Colab "userdata")
login(token=userdata.get("HF_Token"), add_to_git_credential=True)
login(new_session=False)

import os, gc, re, time, torch, pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Device:", device)

# ============================================================
# 🧩 STEP 2: Load CommonsenseQA Dataset (100 samples)
# ============================================================
print("\n📦 Loading CommonsenseQA dataset...")
dataset = load_dataset("commonsense_qa")
test_data = dataset["validation"].select(range(100))  # only 100 rows
print(f"✅ Number of samples used: {len(test_data)}")
print("✅ Columns:", test_data.column_names)

# ============================================================
# 🧩 STEP 3: Helper Functions
# ============================================================
def extract_option_answer(text):
    """Extracts the final letter option (a-e) from model output."""
    match = re.search(r"\b([a-eA-E])\b", text)
    if match:
        return match.group(1).lower()
    return text.strip().lower()

def calculate_accuracy(preds, refs):
    correct = sum(p == r for p, r in zip(preds, refs))
    return correct / len(preds)

def compute_metrics(preds, refs):
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")

    rouge_scores = rouge.compute(predictions=preds, references=refs)
    bert_scores = bertscore.compute(predictions=preds, references=refs, lang="en")

    results = {f"ROUGE_{k}": v for k, v in rouge_scores.items()}
    results["BERTScore_F1"] = sum(bert_scores["f1"]) / len(bert_scores["f1"])
    results["Exact_Match"] = calculate_accuracy(preds, refs)
    results["Accuracy"] = results["Exact_Match"]
    return results

# ============================================================
# 🧩 STEP 4: Batched Inference (T4-friendly)
# ============================================================
def run_inference_batched(model_name, dataset, batch_size=2, quantize_4bit=True):
    torch.cuda.empty_cache()
    gc.collect()

    print(f"\n🚀 Running inference for: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True
    }
    if quantize_4bit:
        model_kwargs["load_in_4bit"] = True

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    prompt_prefix = "Choose the correct answer (a/b/c/d/e) without explanation.\nQuestion: "
    inputs, refs = [], []

    for item in dataset:
        question = item["question"]
        labels = item["choices"]["label"]
        texts = item["choices"]["text"]
        options = [f"{l}) {t}" for l, t in zip(labels, texts)]
        correct = item["answerKey"].lower()
        full_prompt = f"{prompt_prefix}{question}\nOptions: {', '.join(options)}\nAnswer:"
        inputs.append(full_prompt)
        refs.append(correct)

    preds = []
    start_time = time.time()

    for i in tqdm(range(0, len(inputs), batch_size), desc="🔮 Generating answers (batched)"):
        batch_inputs = inputs[i:i+batch_size]
        tokenized = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**tokenized, max_new_tokens=32)
        for out in outputs:
            pred = tokenizer.decode(out, skip_special_tokens=True)
            preds.append(extract_option_answer(pred))

    inference_time = time.time() - start_time
    results = compute_metrics(preds, refs)
    results["Inference_Time(s)"] = round(inference_time, 2)
    results["Model_Size(GB)"] = round(sum(p.numel() for p in model.parameters()) * 2 / (1024**3), 2)
    results["Peak_GPU_Memory(GB)"] = round(torch.cuda.max_memory_allocated(device) / (1024**3), 2)

    df_model = pd.DataFrame([results])
    file_name = f"{model_name.replace('/', '_')}_commonsenseqa_results.csv"
    df_model.to_csv(file_name, index=False)
    print(f"✅ Saved results for {model_name} to {file_name}")

    del model, tokenized, outputs
    torch.cuda.empty_cache()
    gc.collect()
    return results

# ============================================================
# 🧩 STEP 5: Run GPT-2 Models Sequentially
# ============================================================
gpt2_models = [
    ("openai-community/gpt2-medium", 2),  # ~0.4B
    ("openai-community/gpt2-xl", 1),      # ~2B
]

results_all = {}
for model_name, batch_size in gpt2_models:
    results_all[model_name] = run_inference_batched(model_name, test_data, batch_size=batch_size)

# ============================================================
# 🧩 STEP 6: Display Results
# ============================================================
df_results = pd.DataFrame(results_all).T
print("\n📊 Baseline Vanilla Inference Results (100 samples, CommonsenseQA):")
display(df_results)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 18.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Device: cuda

📦 Loading CommonsenseQA dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🔮 Generating answers (batched): 100%|██████████| 100/100 [03:15<00:00,  1.95s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Saved results for openai-community/gpt2-xl to openai-community_gpt2-xl_commonsenseqa_results.csv

📊 Baseline Vanilla Inference Results (100 samples, CommonsenseQA):


,ROUGE_rouge1,ROUGE_rouge2,ROUGE_rougeL,ROUGE_rougeLsum,BERTScore_F1,Exact_Match,Accuracy,Inference_Time(s),Model_Size(GB),Peak_GPU_Memory(GB)
openai-community/gpt2-medium,0.17,0.0,0.17,0.17,0.957603,0.17,0.17,75.17,0.38,1.30
openai-community/gpt2-xl,0.17,0.0,0.17,0.17,0.957603,0.17,0.17,195.19,1.53,2.98
